**Fig. 5C-F and Video S6: fine-tune the zebrafish lateral DSM model on the corrected human crop**

Needs a GPU.

In [ ]:
# Working directory: data/fig5/ under the repository root, where the Fig. 5 deposit folder is downloaded
# (domain-specialized_finetuned_bootstrap/ = the DSM model pair, human_finetuning/ = the fine-tuned pair
# and the exported splits, salivary_gland/; the corrected human crop is read from ../ground_truth/human/crop/).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # the notebook's own folder, for finetune_helpers.py
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig5")
print("working directory:", os.getcwd())


# Lateral

In [ ]:
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

from finetune_helpers import BootstrappedSliceLoader, export_split_dataset   # next to this notebook
from packages.epicycle_training import TrainingConfig, CellposeTrainer   # the EpiCycle plugin

In [ ]:

IMAGE_DIR = Path("../ground_truth/human/crop/img")
MASK_DIR = Path("../ground_truth/human/crop/mask")
IMAGE_GLOB = "*.tif"
MASK_SUFFIX = "_mask.tif"   # image: {name}.tif -> mask: {name}_mask.tif

VIEW = "XY"  # "XY", "XZ", or "YZ"
EDGE_BUFFER = 0  # used for XZ/YZ only
MAX_STACKS = None  # set int to limit number of stack pairs
MAX_SLICES_PER_STACK = None  # set int to cap slices loaded from each stack

TRAIN_FRAC = 0.8
VAL_FRAC = 0.2
RANDOM_SEED = 42

USE_CUSTOM_PRETRAINED = True
CUSTOM_PRETRAINED_MODEL = Path("domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootstrap4Finetuned_lateral_500epoch_scratch_lenient_rescale_True")

USE_CELLPOSE_BASE = False
BASE_MODEL = "cyto3"

MODEL_SAVE_DIR = Path("human_finetuning/multi_stack_models-finetune_Zf_to_human")
MODEL_NAME = "multi_stack_lateral_finetune_Zf2human"

training_config = TrainingConfig(
    n_epochs=500,
    learning_rate=1e-2,
    batch_size=16,
    channels=[2, 1],
)

if abs((TRAIN_FRAC + VAL_FRAC) - 1.0) > 1e-6:
    raise ValueError("TRAIN_FRAC + VAL_FRAC must equal 1.0")

if USE_CUSTOM_PRETRAINED and USE_CELLPOSE_BASE:
    raise ValueError("Choose either a custom pretrained model or a Cellpose base model.")

In [ ]:
image_paths = sorted(IMAGE_DIR.glob(IMAGE_GLOB))
if MAX_STACKS is not None:
    image_paths = image_paths[:MAX_STACKS]

if not image_paths:
    raise FileNotFoundError(f"No images matching {IMAGE_GLOB} in {IMAGE_DIR}")

stack_pairs = []
missing_masks = []

for image_path in image_paths:
    stack_name = image_path.stem
    mask_path = MASK_DIR / f"{stack_name}{MASK_SUFFIX}"
    if mask_path.exists():
        stack_pairs.append((image_path, mask_path))
    else:
        missing_masks.append(mask_path)

print(f"Paired stacks: {len(stack_pairs)}")
if missing_masks:
    print(f"Missing masks: {len(missing_masks)}")
    for p in missing_masks[:10]:
        print(f"  - {p}")
    if len(missing_masks) > 10:
        print(f"  ... and {len(missing_masks) - 10} more")

if not stack_pairs:
    raise FileNotFoundError("No valid image/mask pairs were found.")

Paired stacks: 1


In [ ]:
imgs = []
masks = []
stack_slice_counts = {}

for idx, (image_path, mask_path) in enumerate(stack_pairs, start=1):
    print(f"[{idx}/{len(stack_pairs)}] Loading {image_path.name}")

    loader = BootstrappedSliceLoader(
        image_path=image_path,
        mask_path=mask_path,
        view=VIEW,
        relabel_masks=True,
        min_size=100,
        smoothing_disk_size=1,
        edge_buffer=EDGE_BUFFER,
    )

    indices = list(range(len(loader)))
    if MAX_SLICES_PER_STACK is not None:
        indices = indices[:MAX_SLICES_PER_STACK]

    for i in indices:
        img, mask = loader.get_slice(i)
        imgs.append(img)
        masks.append(mask)

    stack_slice_counts[image_path.stem] = len(indices)
    print(f"  Added {len(indices)} slices")

print(f"\nTotal slices loaded: {len(imgs)}")

if len(imgs) < 5:
    raise ValueError("Too few slices loaded to train reliably. Check paths or reduce filtering.")
     

[1/1] Loading human_patch1.tif
  Added 159 slices

Total slices loaded: 159


In [ ]:

n = len(imgs)
rng = np.random.default_rng(RANDOM_SEED)
perm = rng.permutation(n)

train_count = int(TRAIN_FRAC * n)
val_count = int(VAL_FRAC * n)
test_count = n - train_count - val_count

if train_count == 0 or val_count == 0:
    raise ValueError(f"Split produced empty set: train={train_count}, val={val_count}")

train_idx = perm[:train_count]
val_idx = perm[train_count:train_count + val_count]
test_idx = perm[train_count + val_count:]

train_imgs = [imgs[i] for i in train_idx]
train_masks = [masks[i] for i in train_idx]
val_imgs = [imgs[i] for i in val_idx]
val_masks = [masks[i] for i in val_idx]
test_imgs = [imgs[i] for i in test_idx]
test_masks = [masks[i] for i in test_idx]

print(f"Train: {len(train_imgs)}, Val: {len(val_imgs)}, Test: {len(test_imgs)}")

Train: 127, Val: 31, Test: 1


In [ ]:
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

pretrained_model = CUSTOM_PRETRAINED_MODEL if USE_CUSTOM_PRETRAINED else None
base_model = BASE_MODEL if USE_CELLPOSE_BASE else None

trainer = CellposeTrainer(training_config)
# The trainer writes MODEL_SAVE_DIR/models/MODEL_NAME and its loss curve next to it. As in the Fig. 2
# round loop, a model with its loss curve is kept: a rerun then overwrites neither the deposited model
# nor its split export in the next cell (a checkpoint without its loss curve is from a killed run and
# is trained again).
model_path = MODEL_SAVE_DIR / "models" / MODEL_NAME
trained = False
if model_path.exists() and model_path.with_name(f"{MODEL_NAME}_train_loss.npy").exists():
    print(f"model already exists at {model_path}: delete it to retrain")
else:
    _ = trainer.fit(                       # the trainer's repr is not a result
        train_imgs,
        train_masks,
        val_data=(val_imgs, val_masks),
        pretrained_model=pretrained_model,
        base_model=base_model,
        model_name=MODEL_NAME,
        save_path=MODEL_SAVE_DIR,
    )
    trained = True

Fine-tuning from custom model: models/domain-specialized_model_benchmarking/domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootstrap4Finetuned_lateral_500epoch_scratch_lenient_rescale_True
Training model: multi_stack_lateral_finetune_Zf2human
  Epochs: 500
  Learning rate: 0.01
  Batch size: 16
  Training samples: 127
  Validation samples: 31


  0%|                                                                   | 0/127 [00:00<?, ?it/s]empty masks!
empty masks!
 10%|█████▉                                                    | 13/127 [00:00<00:05, 19.21it/s]empty masks!
empty masks!
 41%|███████████████████████▋                                  | 52/127 [00:02<00:03, 20.01it/s]empty masks!
empty masks!
 55%|████████████████████████████████▎                          | 17/31 [00:01<00:01, 13.56it/s]empty masks!
empty masks!
  0%|                                                                   | 0/127 [00:00<?, ?it/s]<env>/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
<env>/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
Epochs: 100%|█████████████████████████████████████████████████| 500/500 [24:26<00:00,  2.93s/it]

Training config saved to: models/multi_stack_models-finetune_Zf_to_human/models/multi_stack_lateral_finetune_Zf2human_training_config.json
Training complete. Model saved to: models/multi_stack_models-finetune_Zf_to_human/models/multi_stack_lateral_finetune_Zf2human


In [ ]:
if not trained:
    print("split export skipped: the model already existed")
else:
    export_split_dataset(
        train_imgs,
        train_masks,
        val_imgs,
        val_masks,
        output_dir="human_finetuning/dense_FT_ZfHuman-lateral",
        test_imgs=test_imgs,
        test_masks=test_masks,
        metadata={
            "view": VIEW,
            "image_dir": str(IMAGE_DIR),
            "mask_dir": str(MASK_DIR),
            "n_stacks": len(stack_pairs),
            "stack_slice_counts": stack_slice_counts,
        },
        prefix="multistack",
    )

Exported dataset to human_data\dense_FT_ZfHuman-lateral
  Train: 127 samples
  Val: 31 samples
  Test: 1 samples
